# 15. 모델 최적화와 실전 배포

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_15_%EB%AA%A8%EB%8D%B8_%EC%B5%9C%EC%A0%81%ED%99%94%EC%99%80_%EC%8B%A4%EC%A0%84_%EB%B0%B0%ED%8F%AC.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 실습: PTQ 양자화와 지식 증류

### 🔬 실습 — 훈련 후 동적 양자화(PTQ)와 크기·속도 비교

① **기준 모델 준비**: 학습된 모델을 준비하고 기준선을 측정한다.


In [ ]:
import torch, torch.nn as nn, os, time
model = nn.Sequential(nn.Linear(128, 256), nn.ReLU(),
                      nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 10))
model.eval()
torch.save(model.state_dict(), "fp32.pt")
print("FP32 크기(KB):", round(os.path.getsize("fp32.pt")/1024, 1))

② **동적 양자화**: Linear 계층을 INT8로 변환한다.

> 참고: macOS ARM(Apple Silicon) 로컬에서는 실행 전 `torch.backends.quantized.engine = 'qnnpack'`을 지정해야 한다(기본 엔진 미설정 시 NoQEngine 오류). Colab 등 x86 CPU에서는 그대로 동작한다.


In [ ]:
q_model = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8)
torch.save(q_model.state_dict(), "int8.pt")
print("INT8 크기(KB):", round(os.path.getsize("int8.pt")/1024, 1))

③ **추론 속도 비교**: 동일 입력으로 평균 추론 시간을 측정한다.


In [ ]:
def bench(m, n=200):
    x = torch.randn(1, 128)
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n): m(x)
    return (time.time() - t0) / n * 1000     # ms/추론
print(f"FP32 {bench(model):.3f} ms | INT8 {bench(q_model):.3f} ms")

**실행 결과 예시**

```
FP32 크기(KB): 198.4
INT8 크기(KB): 54.3
FP32 0.182 ms | INT8 0.121 ms
```

(크기는 코드 그대로의 실행 결과로, 약 1/4 수준이다. 정확히는 3.7분의 1인데, 가중치는 1/4이 되지만 스케일·제로포인트 같은 양자화 정보가 더해져 4분의 1보다 조금 덜 줄기 때문이다. 시간은 x86 CPU에서의 예시이며, 프로세서 종류와 모델 크기에 따라 INT8이 오히려 느려질 수도 있다. 메모리는 확실히 줄지만 속도는 실제 타깃 장비에서 재 봐야 한다.)


### 🔬 실습 — 지식 증류 학습 루프

① **KD 손실함수 정의**: 정답 CE와 교사 모방 KL(온도 \(T\))을 합해 정의한다.


In [ ]:
import torch.nn.functional as F
def kd_loss(student_logits, teacher_logits, y, T=4.0, alpha=0.5):
    ce = F.cross_entropy(student_logits, y)
    kl = F.kl_div(F.log_softmax(student_logits / T, dim=1),
                  F.softmax(teacher_logits / T, dim=1),
                  reduction="batchmean") * (T * T)     # T^2 보상
    return (1 - alpha) * ce + alpha * kl

② **학습 스텝**: 교사는 고정(eval), 학생만 학습한다.


In [ ]:
# --- (실습용) 교사/학생 모델과 합성 분류 데이터 준비 ---
# setup 셀의 torch, 셀 3의 nn을 사용합니다. 교사(큰 모델)/학생(작은 모델), 10-클래스 분류.
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
teacher = nn.Sequential(nn.Linear(128, 256), nn.ReLU(),
                        nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
student = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
_N = 256
_X_kd = torch.randn(_N, 128)
_y_kd = torch.randint(0, 10, (_N,))
train_loader = DataLoader(TensorDataset(_X_kd, _y_kd), batch_size=64, shuffle=True)


In [ ]:
teacher.eval()                                # 교사 고정
optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)
for x, y in train_loader:
    with torch.no_grad():
        t_logits = teacher(x)
    s_logits = student(x)
    loss = kd_loss(s_logits, t_logits, y, T=4.0, alpha=0.5)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

**실행 결과 예시**

```
교사 정확도 0.94 (파라미터 11.2M)
학생 정확도 0.91 (파라미터 1.3M)  <- 단독 학습할 때 0.86
```

학생이 교사에 근접하며 크기는 훨씬 작다. (수치는 개념 예시이다 — 위 합성 데이터 구성에서는 교사·학생 구성에 따라 달라진다.)


---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
